In [1]:
# =================================================================================================
# NHANES 2017-2018 LABORATORY
# INSULIN - INS_J
#
# INPUT:
#   INS_J.xpt
#
# OUTPUT:
#   INS_J_corrected.csv
#
#
# COMPONENT:
#   Laboratory
#
# SUBCOMPONENT:
#   Insulin
#
#
# ATTACHED SOURCE FINDINGS
# -------------------------------------------------------------------------------------------------
#
# Rows                         = 3,036
# Columns                      = 5
#
# Variables:
#
#   SEQN
#   WTSAF2YR
#   LBXIN
#   LBDINSI
#   LBDINLC
#
# SEQN range                   = 93708-102956
# Missing SEQN                 = 0
# Duplicate SEQN               = 0
#
#
# VALIDATED SAS XPORT TINY-ZERO ARTIFACTS
# -------------------------------------------------------------------------------------------------
#
# WTSAF2YR       325
# LBDINLC      2,818
#
# TOTAL         3,143
#
#
# CDC WTSAF2YR
# -------------------------------------------------------------------------------------------------
#
# Positive fasting weights = 2,711
# Zero                     =   325
# Missing                  =     0
#
# Actual attached XPT positive range:
#
#   minimum = 9133.518063
#   maximum = 944153.249752
#
# CDC displayed range:
#
#   minimum = 9133.518063
#   maximum = 944153.24975
#
# CDC display precision is slightly shorter than the underlying
# XPT precision. DO NOT round the source value.
#
#
# CDC INSULIN
# -------------------------------------------------------------------------------------------------
#
# LBXIN:
#
#   results = 2,825
#   missing =   211
#   range   = 0.71 - 485.1
#
#
# LBDINSI:
#
#   results = 2,825
#   missing =   211
#   range   = 4.26 - 2910.6
#
#
# LBDINLC:
#
#   0 = detectable result         2,818
#   1 = below detectable limit        7
#   missing                         211
#
#
# IMPORTANT 2017-2018 DESIGN
# -------------------------------------------------------------------------------------------------
#
# INS_J DOES NOT contain:
#
#   PHAFSTHR
#   PHAFSTMN
#
# CDC states these should be obtained from FASTQX_J.
#
#
# GENUINE DECIMAL VARIABLES
# -------------------------------------------------------------------------------------------------
#
# WTSAF2YR
# LBXIN
# LBDINSI
#
#
# PROVEN INTEGRAL/CODE VARIABLES
# -------------------------------------------------------------------------------------------------
#
# SEQN
# LBDINLC
#
#
# FIDELITY RULES
# -------------------------------------------------------------------------------------------------
#
# 1. Restore ONLY validated SAS XPORT tiny-zero values.
#
# 2. Never globally replace tiny numbers.
#
# 3. Preserve genuine Lab decimals exactly.
#
# 4. Do NOT round.
#
# 5. Do NOT use float_format=.
#
# 6. Convert ONLY proven integral/code variables to nullable Int64.
#
# 7. Preserve missing positions exactly.
#
# 8. Validate CDC distributions.
#
# 9. Validate exact numeric XPT -> CSV round-trip fidelity.
#
# 10. Ensure SEQN does not contain ".0".
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


LAB_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Labs"
)


XPT_FILE = (
    LAB_DIR
    / "INS_J.xpt"
)


CSV_FILE = (
    LAB_DIR
    / "INS_J_corrected.csv"
)


# =================================================================================================
# 3. VERIFY INPUT FILE EXISTS
# =================================================================================================

if not XPT_FILE.exists():

    raise FileNotFoundError(
        f"INS_J XPT file not found:\n{XPT_FILE}"
    )


# =================================================================================================
# 4. READ ORIGINAL XPT
# =================================================================================================

df = pd.read_sas(
    XPT_FILE,
    format="xport",
    encoding="latin1"
)


#
# Preserve untouched decoded source.
#

source_df = df.copy()


print("=" * 110)

print(
    "NHANES 2017-2018 LABORATORY - INS_J"
)

print(
    "INSULIN XPT -> FIDELITY-PRESERVING CSV"
)

print("=" * 110)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Variables:"
)


print(
    df.columns.tolist()
)


# =================================================================================================
# 5. EXPECTED STRUCTURE
# =================================================================================================

EXPECTED_COLUMNS = [

    "SEQN",

    "WTSAF2YR",

    "LBXIN",

    "LBDINSI",

    "LBDINLC"

]


EXPECTED_ROWS = 3036


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "INS_J structure differs from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected INS_J row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(df):,}"
    )


print(
    "PASS: INS_J structure = 3,036 rows x 5 columns."
)


# =================================================================================================
# 6. VERIFY ALL VARIABLES LOAD NUMERICALLY
# =================================================================================================

numeric_columns = (

    df.select_dtypes(
        include=[np.number]
    )

    .columns

    .tolist()

)


if numeric_columns != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all five INS_J variables "
        "to load numerically from XPT."
    )


print(
    "PASS: All five variables loaded numerically."
)


# =================================================================================================
# 7. FIELD TYPE PLAN
# =================================================================================================

DECIMAL_COLUMNS = [

    "WTSAF2YR",

    "LBXIN",

    "LBDINSI"

]


INTEGER_COLUMNS = [

    "SEQN",

    "LBDINLC"

]


print(
    "\n--- FIELD TYPE PLAN ---"
)


print(
    "Genuine Lab decimal fields:",
    DECIMAL_COLUMNS
)


print(
    "Integral/code fields:",
    INTEGER_COLUMNS
)


# =================================================================================================
# 8. SEQN SOURCE CHECK
# =================================================================================================

seqn_min = int(
    df["SEQN"].min()
)


seqn_max = int(
    df["SEQN"].max()
)


seqn_missing = int(

    df["SEQN"]
    .isna()
    .sum()

)


seqn_duplicates = int(

    df["SEQN"]
    .duplicated()
    .sum()

)


print(
    "\n--- SEQN SOURCE CHECK ---"
)


print(
    "Minimum:",
    seqn_min
)


print(
    "Maximum:",
    seqn_max
)


print(
    "Missing:",
    seqn_missing
)


print(
    "Duplicates:",
    seqn_duplicates
)


if seqn_min != 93708:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if seqn_max != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if seqn_missing != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if seqn_duplicates != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN range 93708-102956 validated."
)


# =================================================================================================
# 9. VERIFY 2017-2018 INS_J DESIGN
# =================================================================================================

#
# INS_J must NOT contain PHAFSTHR / PHAFSTMN.
#

for col in [

    "PHAFSTHR",

    "PHAFSTMN"

]:

    if col in df.columns:

        raise ValueError(
            f"Unexpected {col} found in INS_J.\n"
            "For 2017-2018 these fields belong to FASTQX_J."
        )


print(
    "PASS: INS_J correctly does NOT contain "
    "PHAFSTHR / PHAFSTMN."
)


# =================================================================================================
# 10. DEFINE SAS XPORT TINY-ZERO VALUE
# =================================================================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


# =================================================================================================
# 11. DISCOVER TINY VALUES
# =================================================================================================

tiny_counts = (

    df[EXPECTED_COLUMNS]

    .eq(TINY_VALUE)

    .sum()

)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPORT TINY-ZERO CHECK ---"
)


print(
    tiny_counts.to_string()
)


print(
    "\nTotal tiny values:",
    f"{tiny_total:,}"
)


# =================================================================================================
# 12. VALIDATED TINY-ZERO PATTERN
# =================================================================================================

EXPECTED_TINY_COUNTS = {

    "WTSAF2YR": 325,

    "LBDINLC": 2818

}


actual_tiny_counts = (

    tiny_counts[
        tiny_counts > 0
    ]

    .astype(int)

    .to_dict()

)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: INS_J tiny-value pattern differs "
        "from validated attached source.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 3143:

    raise ValueError(
        "Unexpected total tiny-value count.\n"
        f"Expected: 3,143\n"
        f"Actual:   {tiny_total:,}"
    )


print(
    "PASS: Exact 3,143-value tiny-zero pattern validated."
)


# =================================================================================================
# 13. ORDINARY ZERO CHECK BEFORE RESTORATION
# =================================================================================================

ordinary_zero_counts = (

    df[EXPECTED_COLUMNS]

    .eq(0)

    .sum()

)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORDINARY ZERO CHECK BEFORE RESTORATION ---"
)


print(
    ordinary_zero_counts.to_string()
)


print(
    "\nTotal ordinary numeric zeros:",
    ordinary_zero_total
)


if ordinary_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros found "
        "before XPORT correction."
    )


print(
    "PASS: Validated zeros are represented by "
    "the SAS XPORT tiny value."
)


# =================================================================================================
# 14. BUILD EXPECTED CORRECTED SOURCE
# =================================================================================================

#
# This is the scientific target:
#
# original source
#
# +
#
# ONLY validated tiny-zero restorations.
#

expected_df = (
    source_df.copy()
)


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[

        expected_df[col]
        .eq(TINY_VALUE),

        col

    ] = 0


# =================================================================================================
# 15. RESTORE ONLY VALIDATED TRUE ZEROS
# =================================================================================================

correction_counts = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: tiny-value count changed.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_counts[
        col
    ] = actual_count


total_corrections = sum(
    correction_counts.values()
)


print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


for col, count in correction_counts.items():

    print(
        f"{col}: {count:,}"
    )


print(
    "Total:",
    f"{total_corrections:,}"
)


if total_corrections != 3143:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "PASS: 3,143 validated zeros restored."
)


# =================================================================================================
# 16. VERIFY NO TINY VALUES REMAIN
# =================================================================================================

remaining_tiny = int(

    df[EXPECTED_COLUMNS]

    .eq(TINY_VALUE)

    .sum()

    .sum()

)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# =================================================================================================
# 17. VERIFY CORRECTED ZERO DISTRIBUTION
# =================================================================================================

corrected_zero_counts = {}


for col in EXPECTED_COLUMNS:

    count = int(

        df[col]
        .eq(0)
        .sum()

    )


    if count > 0:

        corrected_zero_counts[
            col
        ] = count


print(
    "\n--- CORRECTED TRUE-ZERO DISTRIBUTION ---"
)


print(
    corrected_zero_counts
)


if corrected_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Corrected zero distribution differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{corrected_zero_counts}"
    )


print(
    "PASS: Corrected zero distribution validated."
)


# =================================================================================================
# 18. CDC CHECK - WTSAF2YR COUNTS
# =================================================================================================

wtsaf = (
    df["WTSAF2YR"]
)


wtsaf_positive = int(

    (
        wtsaf > 0
    )

    .sum()

)


wtsaf_zero = int(

    wtsaf
    .eq(0)
    .sum()

)


wtsaf_missing = int(

    wtsaf
    .isna()
    .sum()

)


print(
    "\n--- CDC CHECK: WTSAF2YR ---"
)


print(
    "Positive:",
    f"{wtsaf_positive:,}"
)


print(
    "Zero:",
    f"{wtsaf_zero:,}"
)


print(
    "Missing:",
    wtsaf_missing
)


if wtsaf_positive != 2711:

    raise ValueError(
        "WTSAF2YR positive count differs from CDC."
    )


if wtsaf_zero != 325:

    raise ValueError(
        "WTSAF2YR zero count differs from CDC."
    )


if wtsaf_missing != 0:

    raise ValueError(
        "WTSAF2YR missing count differs from CDC."
    )


print(
    "PASS: WTSAF2YR counts match CDC."
)


# =================================================================================================
# 19. CDC WTSAF2YR DISPLAY-RANGE CHECK
# =================================================================================================

#
# Actual attached XPT:
#
#   minimum positive = 9133.518063
#   maximum positive = 944153.249752
#
# CDC displays:
#
#   minimum = 9133.518063
#   maximum = 944153.24975
#
# CDC display precision is slightly shorter.
#
# This tolerance is ONLY for comparison to the web codebook.
#
# Actual source fidelity is checked later with rtol=0 / atol=0.
#

positive_wtsaf = (

    wtsaf.loc[
        wtsaf > 0
    ]

)


actual_wtsaf_min = float(
    positive_wtsaf.min()
)


actual_wtsaf_max = float(
    positive_wtsaf.max()
)


CDC_WTSAF2YR_MIN = 9133.518063

CDC_WTSAF2YR_MAX = 944153.24975


print(
    "\n--- WTSAF2YR RANGE CHECK ---"
)


print(
    "Actual XPT minimum:",
    repr(actual_wtsaf_min)
)


print(
    "CDC displayed minimum:",
    CDC_WTSAF2YR_MIN
)


print(
    "Actual XPT maximum:",
    repr(actual_wtsaf_max)
)


print(
    "CDC displayed maximum:",
    CDC_WTSAF2YR_MAX
)


if not np.isclose(

    actual_wtsaf_min,

    CDC_WTSAF2YR_MIN,

    rtol=0,

    atol=1e-6

):

    raise ValueError(
        "WTSAF2YR positive minimum differs "
        "from CDC beyond display tolerance."
    )


if not np.isclose(

    actual_wtsaf_max,

    CDC_WTSAF2YR_MAX,

    rtol=0,

    atol=1e-5

):

    raise ValueError(
        "WTSAF2YR positive maximum differs "
        "from CDC beyond display tolerance."
    )


print(
    "PASS: WTSAF2YR range agrees with CDC "
    "displayed values."
)


# =================================================================================================
# 20. CDC CHECK - LBXIN
# =================================================================================================

lbxin_nonmissing = int(

    df["LBXIN"]
    .notna()
    .sum()

)


lbxin_missing = int(

    df["LBXIN"]
    .isna()
    .sum()

)


lbxin_min = float(
    df["LBXIN"].min()
)


lbxin_max = float(
    df["LBXIN"].max()
)


print(
    "\n--- CDC CHECK: LBXIN ---"
)


print(
    "Nonmissing:",
    f"{lbxin_nonmissing:,}"
)


print(
    "Missing:",
    lbxin_missing
)


print(
    "Range:",
    lbxin_min,
    "to",
    lbxin_max
)


if lbxin_nonmissing != 2825:

    raise ValueError(
        "LBXIN nonmissing count differs from CDC."
    )


if lbxin_missing != 211:

    raise ValueError(
        "LBXIN missing count differs from CDC."
    )


if not np.isclose(

    lbxin_min,

    0.71,

    rtol=0,

    atol=1e-12

):

    raise ValueError(
        "LBXIN minimum differs from CDC."
    )


if not np.isclose(

    lbxin_max,

    485.1,

    rtol=0,

    atol=1e-12

):

    raise ValueError(
        "LBXIN maximum differs from CDC."
    )


print(
    "PASS: LBXIN matches CDC."
)


# =================================================================================================
# 21. CDC CHECK - LBDINSI
# =================================================================================================

lbdinsi_nonmissing = int(

    df["LBDINSI"]
    .notna()
    .sum()

)


lbdinsi_missing = int(

    df["LBDINSI"]
    .isna()
    .sum()

)


lbdinsi_min = float(
    df["LBDINSI"].min()
)


lbdinsi_max = float(
    df["LBDINSI"].max()
)


print(
    "\n--- CDC CHECK: LBDINSI ---"
)


print(
    "Nonmissing:",
    f"{lbdinsi_nonmissing:,}"
)


print(
    "Missing:",
    lbdinsi_missing
)


print(
    "Range:",
    lbdinsi_min,
    "to",
    lbdinsi_max
)


if lbdinsi_nonmissing != 2825:

    raise ValueError(
        "LBDINSI nonmissing count differs from CDC."
    )


if lbdinsi_missing != 211:

    raise ValueError(
        "LBDINSI missing count differs from CDC."
    )


if not np.isclose(

    lbdinsi_min,

    4.26,

    rtol=0,

    atol=1e-12

):

    raise ValueError(
        "LBDINSI minimum differs from CDC."
    )


if not np.isclose(

    lbdinsi_max,

    2910.6,

    rtol=0,

    atol=1e-12

):

    raise ValueError(
        "LBDINSI maximum differs from CDC."
    )


print(
    "PASS: LBDINSI matches CDC."
)


# =================================================================================================
# 22. CDC CHECK - LBDINLC
# =================================================================================================

#
# CDC:
#
# 0 = detectable result          2,818
# 1 = below detectable limit         7
# Missing                          211
#

lbdinlc_counts = (

    df["LBDINLC"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


lbdinlc_missing = int(

    df["LBDINLC"]

    .isna()

    .sum()

)


EXPECTED_LBDINLC = {

    0.0: 2818,

    1.0: 7

}


print(
    "\n--- CDC CHECK: LBDINLC ---"
)


print(
    lbdinlc_counts
)


print(
    "Missing:",
    lbdinlc_missing
)


if lbdinlc_counts != EXPECTED_LBDINLC:

    raise ValueError(
        "LBDINLC distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_LBDINLC}\n\n"
        f"Actual:\n{lbdinlc_counts}"
    )


if lbdinlc_missing != 211:

    raise ValueError(
        "LBDINLC missing count differs from CDC."
    )


print(
    "PASS: LBDINLC matches CDC exactly."
)


# =================================================================================================
# 23. FRACTIONAL VALUE DISCOVERY
# =================================================================================================

#
# Diagnostic only.
#
# np.round() does NOT modify the dataframe.
#

fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) == 0:

        count = 0


    else:

        count = int(

            (
                ~np.isclose(

                    values,

                    np.round(
                        values
                    ),

                    rtol=0,

                    atol=1e-12

                )
            )

            .sum()

        )


    fractional_counts[
        col
    ] = count


print(
    "\n--- FRACTIONAL VALUE DISCOVERY ---"
)


for col, count in fractional_counts.items():

    print(
        f"{col}: {count:,}"
    )


#
# Attached source findings:
#
# WTSAF2YR = 2,711 fractional
# LBXIN    = 2,792 fractional
# LBDINSI  = 2,765 fractional
#

EXPECTED_FRACTIONAL_COUNTS = {

    "SEQN": 0,

    "WTSAF2YR": 2711,

    "LBXIN": 2792,

    "LBDINSI": 2765,

    "LBDINLC": 0

}


if fractional_counts != EXPECTED_FRACTIONAL_COUNTS:

    raise ValueError(
        "Fractional-value pattern differs from "
        "validated attached INS_J source.\n\n"
        f"Expected:\n{EXPECTED_FRACTIONAL_COUNTS}\n\n"
        f"Actual:\n{fractional_counts}"
    )


print(
    "PASS: Genuine decimal/integral classifications validated."
)


# =================================================================================================
# 24. VERIFY ONLY VALIDATED VALUES CHANGED
# =================================================================================================

source_change_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        df[col]

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        expected_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        source_change_failures.append(
            col
        )


if source_change_failures:

    raise ValueError(
        "Unexpected source-value changes:\n"
        + str(
            source_change_failures
        )
    )


print(
    "PASS: Only validated tiny-zero values changed."
)


# =================================================================================================
# 25. MISSING-POSITION FIDELITY
# =================================================================================================

missing_position_errors = 0


for col in EXPECTED_COLUMNS:

    source_missing = (

        source_df[col]

        .isna()

        .to_numpy()

    )


    corrected_missing = (

        df[col]

        .isna()

        .to_numpy()

    )


    missing_position_errors += int(

        (
            source_missing
            !=
            corrected_missing
        )

        .sum()

    )


if missing_position_errors != 0:

    raise ValueError(
        "Missing-value positions changed during correction."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# =================================================================================================
# 26. CONVERT ONLY PROVEN INTEGRAL/CODE FIELDS
# =================================================================================================

#
# This removes unwanted:
#
#   93708.0 -> 93708
#
# and:
#
#   0.0 -> 0
#
#
# DO NOT convert:
#
#   WTSAF2YR
#   LBXIN
#   LBDINSI
#

for col in INTEGER_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) > 0:

        integral_mask = np.isclose(

            values,

            np.round(
                values
            ),

            rtol=0,

            atol=1e-12

        )


        if not integral_mask.all():

            raise ValueError(
                f"{col} contains unexpected fractional values."
            )


    df[col] = (

        df[col]

        .astype("Int64")

    )


print(
    "PASS: Only SEQN and LBDINLC converted to Int64."
)


# =================================================================================================
# 27. PRE-EXPORT NUMERIC FIDELITY
# =================================================================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        pd.to_numeric(
            df[col],
            errors="coerce"
        )

        .to_numpy(
            dtype=float,
            na_value=np.nan
        )

    )


    same = np.allclose(

        expected_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Pre-export numeric fidelity failures:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export numeric fidelity."
)


# =================================================================================================
# 28. EXPORT CORRECTED CSV
# =================================================================================================

#
# CRITICAL:
#
# NO:
#
#     float_format=
#
# NO:
#
#     round()
#
# Genuine Lab decimals remain untouched.
#

df.to_csv(

    CSV_FILE,

    index=False,

    na_rep=""

)


print(
    "\nCorrected Laboratory CSV created:"
)


print(
    CSV_FILE
)


# =================================================================================================
# 29. RE-READ WITH ROUND-TRIP FLOAT PARSER
# =================================================================================================

roundtrip_df = pd.read_csv(

    CSV_FILE,

    low_memory=False,

    float_precision="round_trip"

)


if roundtrip_df.shape != (

    EXPECTED_ROWS,

    len(EXPECTED_COLUMNS)

):

    raise ValueError(
        "Final CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "Final CSV column order changed."
    )


print(
    "PASS: Final CSV structure preserved."
)


# =================================================================================================
# 30. EXACT XPT -> CSV NUMERIC ROUND-TRIP
# =================================================================================================

#
# THIS is the strict scientific source-fidelity test.
#
# No tolerance:
#
#   rtol = 0
#   atol = 0
#

roundtrip_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    final_values = (

        roundtrip_df[col]

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        expected_values,

        final_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        roundtrip_failures.append(
            col
        )


if roundtrip_failures:

    raise ValueError(
        "XPT -> CSV exact numeric fidelity failures:\n"
        + str(
            roundtrip_failures
        )
    )


print(
    "PASS: All five fields round-trip numerically exactly."
)


# =================================================================================================
# 31. READ FINAL CSV AS TEXT
# =================================================================================================

raw_csv = pd.read_csv(

    CSV_FILE,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 32. CHECK UNWANTED ".0"
# =================================================================================================

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    bad_mask = (

        raw_csv[col]
        .ne("")

        &

        raw_csv[col]
        .str.endswith(".0")

    )


    bad_count = int(
        bad_mask.sum()
    )


    if bad_count > 0:

        dot_zero_errors[
            col
        ] = bad_count


if dot_zero_errors:

    raise ValueError(
        "Unnecessary '.0' formatting remains:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' in SEQN/code fields."
)


# =================================================================================================
# 33. FINAL SEQN CHECK
# =================================================================================================

if (
    raw_csv["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if (
    raw_csv["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV contains SEQN '.0'."
    )


if (

    ~raw_csv["SEQN"]
    .str.fullmatch(
        r"\d+"
    )

).any():

    raise ValueError(
        "Final CSV contains invalid SEQN formatting."
    )


final_seqn = pd.to_numeric(

    raw_csv["SEQN"],

    errors="raise"

)


if final_seqn.min() != 93708:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


print(
    "PASS: Final SEQN clean."
)


# =================================================================================================
# 34. FINAL TINY-XPORT CHECK
# =================================================================================================

final_tiny_total = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    final_tiny_total += int(

        values
        .eq(TINY_VALUE)
        .sum()

    )


if final_tiny_total != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts in final CSV."
)


# =================================================================================================
# 35. FINAL TRUE-ZERO DISTRIBUTION
# =================================================================================================

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    count = int(

        values
        .eq(0)
        .sum()

    )


    if count > 0:

        final_zero_counts[
            col
        ] = count


print(
    "\n--- FINAL TRUE-ZERO DISTRIBUTION ---"
)


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero distribution differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Exactly 3,143 validated true zeros remain."
)


# =================================================================================================
# 36. SCIENTIFIC-NOTATION CHECK
# =================================================================================================

scientific_notation_fields = {}


for col in EXPECTED_COLUMNS:

    count = int(

        raw_csv[col]

        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )

        .sum()

    )


    if count > 0:

        scientific_notation_fields[
            col
        ] = count


if scientific_notation_fields:

    raise ValueError(
        "Unexpected scientific notation remains:\n"
        + str(
            scientific_notation_fields
        )
    )


print(
    "PASS: No unexpected scientific notation."
)


# =================================================================================================
# 37. FINAL MISSING-POSITION FIDELITY
# =================================================================================================

final_missing_errors = 0


for col in EXPECTED_COLUMNS:

    source_missing = (

        source_df[col]

        .isna()

        .to_numpy()

    )


    final_missing = (

        raw_csv[col]

        .eq("")

        .to_numpy()

    )


    final_missing_errors += int(

        (
            source_missing
            !=
            final_missing
        )

        .sum()

    )


if final_missing_errors != 0:

    raise ValueError(
        "Final CSV missing positions differ from source XPT."
    )


print(
    "PASS: Missing positions preserved through CSV export."
)


# =================================================================================================
# 38. FINAL DECIMAL PRESERVATION CHECK
# =================================================================================================

print(
    "\n--- LAB DECIMAL PRESERVATION ---"
)


for col in DECIMAL_COLUMNS:

    populated_count = int(

        raw_csv[col]
        .ne("")
        .sum()

    )


    decimal_token_count = int(

        (
            raw_csv[col].ne("")
            &
            raw_csv[col]
            .str.contains(
                ".",
                regex=False
            )
        )

        .sum()

    )


    print(
        f"{col}: "
        f"populated={populated_count:,}, "
        f"decimal tokens={decimal_token_count:,}"
    )


    if decimal_token_count == 0:

        raise ValueError(
            f"{col} appears to have been integerized."
        )


print(
    "PASS: Genuine Lab decimal fields remain decimal."
)


# =================================================================================================
# 39. FINAL WTSAF2YR CHECK
# =================================================================================================

final_wtsaf = pd.to_numeric(

    raw_csv["WTSAF2YR"],

    errors="raise"

)


final_positive = int(

    (
        final_wtsaf > 0
    )

    .sum()

)


final_zero = int(

    final_wtsaf
    .eq(0)
    .sum()

)


if final_positive != 2711:

    raise ValueError(
        "Final WTSAF2YR positive count differs."
    )


if final_zero != 325:

    raise ValueError(
        "Final WTSAF2YR zero count differs."
    )


print(
    "PASS: Final WTSAF2YR = "
    "2,711 positive + 325 zero."
)


# =================================================================================================
# 40. FINAL INSULIN RESULT CHECK
# =================================================================================================

if int(

    raw_csv["LBXIN"]
    .ne("")
    .sum()

) != 2825:

    raise ValueError(
        "Final LBXIN count differs."
    )


if int(

    raw_csv["LBDINSI"]
    .ne("")
    .sum()

) != 2825:

    raise ValueError(
        "Final LBDINSI count differs."
    )


print(
    "PASS: Final insulin result counts = 2,825."
)


# =================================================================================================
# 41. FINAL LBDINLC CHECK
# =================================================================================================

final_lbdinlc = (

    raw_csv["LBDINLC"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


EXPECTED_FINAL_LBDINLC = {

    "0": 2818,

    "1": 7,

    "": 211

}


if final_lbdinlc != EXPECTED_FINAL_LBDINLC:

    raise ValueError(
        "Final LBDINLC distribution differs.\n\n"
        f"Expected:\n{EXPECTED_FINAL_LBDINLC}\n\n"
        f"Actual:\n{final_lbdinlc}"
    )


print(
    "PASS: Final LBDINLC:"
)


print(
    "  0 = 2,818"
)


print(
    "  1 = 7"
)


print(
    "  Missing = 211"
)


# =================================================================================================
# 42. FINAL SOURCE-PRECISION WTSAF2YR CHECK
# =================================================================================================

final_positive_wtsaf = (

    final_wtsaf.loc[
        final_wtsaf > 0
    ]

)


final_wtsaf_min = float(
    final_positive_wtsaf.min()
)


final_wtsaf_max = float(
    final_positive_wtsaf.max()
)


print(
    "\n--- FINAL WTSAF2YR SOURCE-PRECISION CHECK ---"
)


print(
    "Original corrected XPT minimum:",
    repr(actual_wtsaf_min)
)


print(
    "Final CSV minimum:",
    repr(final_wtsaf_min)
)


print(
    "Original corrected XPT maximum:",
    repr(actual_wtsaf_max)
)


print(
    "Final CSV maximum:",
    repr(final_wtsaf_max)
)


if final_wtsaf_min != actual_wtsaf_min:

    raise ValueError(
        "WTSAF2YR minimum changed during CSV export."
    )


if final_wtsaf_max != actual_wtsaf_max:

    raise ValueError(
        "WTSAF2YR maximum changed during CSV export."
    )


print(
    "PASS: WTSAF2YR source precision preserved."
)


# =================================================================================================
# 43. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2017-2018 LABORATORY INS_J CONVERSION COMPLETE"
)


print(
    "=" * 110
)


print(
    "Component:"
)


print(
    "  Laboratory"
)


print(
    ""
)


print(
    "Lab file:"
)


print(
    "  INS_J - Insulin"
)


print(
    ""
)


print(
    "Cycle:"
)


print(
    "  2017-2018"
)


print(
    ""
)


print(
    "Rows:",
    f"{len(raw_csv):,}"
)


print(
    "Columns:",
    len(raw_csv.columns)
)


print(
    "Unique SEQN:",
    f"{raw_csv['SEQN'].nunique():,}"
)


print(
    "SEQN range:"
)


print(
    "  93708-102956"
)


print(
    ""
)


print(
    "Validated XPORT zero restorations:"
)


print(
    "  WTSAF2YR =   325"
)


print(
    "  LBDINLC  = 2,818"
)


print(
    "  ----------------"
)


print(
    "  TOTAL     = 3,143"
)


print(
    ""
)


print(
    "Genuine Lab decimal fields preserved:"
)


print(
    "  WTSAF2YR"
)


print(
    "  LBXIN"
)


print(
    "  LBDINSI"
)


print(
    ""
)


print(
    "Integral/code fields cleaned:"
)


print(
    "  SEQN"
)


print(
    "  LBDINLC"
)


print(
    ""
)


print(
    "WTSAF2YR:"
)


print(
    "  Positive = 2,711"
)


print(
    "  Zero     = 325"
)


print(
    "  XPT minimum:",
    repr(actual_wtsaf_min)
)


print(
    "  XPT maximum:",
    repr(actual_wtsaf_max)
)


print(
    ""
)


print(
    "LBXIN:"
)


print(
    "  Results = 2,825"
)


print(
    "  Missing = 211"
)


print(
    "  Range   = 0.71-485.1"
)


print(
    ""
)


print(
    "LBDINSI:"
)


print(
    "  Results = 2,825"
)


print(
    "  Missing = 211"
)


print(
    "  Range   = 4.26-2910.6"
)


print(
    ""
)


print(
    "LBDINLC:"
)


print(
    "  0 = 2,818"
)


print(
    "  1 = 7"
)


print(
    "  Missing = 211"
)


print(
    ""
)


print(
    "PHAFSTHR / PHAFSTMN:"
)


print(
    "  NOT present in INS_J"
)


print(
    "  Use FASTQX_J in unified Lab merge"
)


print(
    ""
)


print(
    "SEQN '.0': 0"
)


print(
    "Duplicate SEQN: 0"
)


print(
    "Remaining tiny XPORT artifacts: 0"
)


print(
    "Unexpected source-value changes: 0"
)


print(
    "Missing-position changes: 0"
)


print(
    "Lab decimal integerization: 0"
)


print(
    "Scientific notation: 0"
)


print(
    "XPT -> CSV exact numeric round-trip: PASS"
)


print(
    "=" * 110
)

NHANES 2017-2018 LABORATORY - INS_J
INSULIN XPT -> FIDELITY-PRESERVING CSV
Rows: 3,036
Columns: 5
Variables:
['SEQN', 'WTSAF2YR', 'LBXIN', 'LBDINSI', 'LBDINLC']
PASS: INS_J structure = 3,036 rows x 5 columns.
PASS: All five variables loaded numerically.

--- FIELD TYPE PLAN ---
Genuine Lab decimal fields: ['WTSAF2YR', 'LBXIN', 'LBDINSI']
Integral/code fields: ['SEQN', 'LBDINLC']

--- SEQN SOURCE CHECK ---
Minimum: 93708
Maximum: 102956
Missing: 0
Duplicates: 0
PASS: SEQN range 93708-102956 validated.
PASS: INS_J correctly does NOT contain PHAFSTHR / PHAFSTMN.

--- ORIGINAL XPORT TINY-ZERO CHECK ---
SEQN           0
WTSAF2YR     325
LBXIN          0
LBDINSI        0
LBDINLC     2818

Total tiny values: 3,143
PASS: Exact 3,143-value tiny-zero pattern validated.

--- ORDINARY ZERO CHECK BEFORE RESTORATION ---
SEQN        0
WTSAF2YR    0
LBXIN       0
LBDINSI     0
LBDINLC     0

Total ordinary numeric zeros: 0
PASS: Validated zeros are represented by the SAS XPORT tiny value.

--- VALIDAT